In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
# Cell 1 — Install & imports
!pip install mlflow xgboost shap evidently -q

import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (roc_auc_score, f1_score, 
                             precision_score, recall_score, 
                             classification_report)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# reproducibility
SEED = 42
np.random.seed(SEED)

print("All imports successful")
print(f"MLflow version: {mlflow.__version__}")
print(f"XGBoost version: {xgb.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 60.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 53.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 41.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 62.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.0/238.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 579.2/579.2 kB 24.1 MB/s eta 0:00:00
   ━━━━

In [3]:
# Cell 2 — Load data
df = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')

print("Shape:", df.shape)
print("\nTarget distribution:")
print(df['isFraud'].value_counts(normalize=True).round(3))
print("\nMemory usage:")
print(f"{df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

Shape: (590540, 394)

Target distribution:
isFraud
0    0.965
1    0.035
Name: proportion, dtype: float64

Memory usage:
2.01 GB


In [4]:
# Cell 4 — Reproducible sklearn Pipeline

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
import pickle

# Step 1 — drop columns with >50% missing (save list for reproducibility)
missing_pct = (df.isnull().sum() / len(df) * 100)
cols_to_drop = missing_pct[missing_pct > 50].index.tolist()
cols_to_drop += ['TransactionID', 'TransactionDT']  # not useful features

df_clean = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns")
print(f"Remaining: {df_clean.shape[1]} columns")

# Step 2 — identify feature types
TARGET = 'isFraud'
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
num_cols = [c for c in df_clean.select_dtypes(include=np.number).columns.tolist() 
            if c != TARGET]

print(f"\nNumeric features: {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)}")

# Step 3 — define preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

# Step 4 — split data BEFORE fitting pipeline (no data leakage)
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=SEED, 
    stratify=y  # preserve fraud ratio in both splits
)

print(f"\nTrain size: {X_train.shape}")
print(f"Val size: {X_val.shape}")
print(f"\nFraud in train: {y_train.mean()*100:.1f}%")
print(f"Fraud in val: {y_val.mean()*100:.1f}%")

# Step 5 — fit preprocessor on train only
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)  # transform only, no fit

# save feature metadata for MLflow logging later
feature_metadata = {
    'dropped_columns': cols_to_drop,
    'numeric_features': num_cols,
    'categorical_features': cat_cols,
    'total_features_used': len(num_cols) + len(cat_cols),
    'train_size': len(X_train),
    'val_size': len(X_val),
    'fraud_rate_train': float(y_train.mean()),
    'fraud_rate_val': float(y_val.mean())
}

print(f"\nPreprocessor fit on train only — no leakage")
print(f"Feature metadata saved for MLflow")
print(f"\nX_train_processed shape: {X_train_processed.shape}")
print(f"X_val_processed shape: {X_val_processed.shape}")

Dropped 176 columns
Remaining: 218 columns

Numeric features: 208
Categorical features: 9

Train size: (472432, 217)
Val size: (118108, 217)

Fraud in train: 3.5%
Fraud in val: 3.5%

Preprocessor fit on train only — no leakage
Feature metadata saved for MLflow

X_train_processed shape: (472432, 217)
X_val_processed shape: (118108, 217)


In [5]:
# Cell 5 — MLflow Setup

import mlflow
import mlflow.xgboost
import mlflow.sklearn
from mlflow.models.signature import infer_signature

# set experiment name
mlflow.set_experiment("fraud-detection")

# test mlflow is working
with mlflow.start_run(run_name="test-run"):
    mlflow.log_param("test", "mlflow working")
    mlflow.log_metric("test_metric", 1.0)

print("MLflow experiment 'fraud-detection' created")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print("\nReady to log experiments")

2026/05/22 09:03:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/22 09:03:11 INFO mlflow.store.db.utils: Updating database tables
2026/05/22 09:03:14 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection' does not exist. Creating a new experiment.


MLflow experiment 'fraud-detection' created
MLflow tracking URI: sqlite:////kaggle/working/mlflow.db

Ready to log experiments


In [6]:
# Cell 6 — MLflow Experiment 1: Logistic Regression Baseline

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

with mlflow.start_run(run_name="logistic-regression-baseline"):
    
    # log feature metadata
    mlflow.log_params(feature_metadata)
    
    # model params
    params = {
        'model_type': 'logistic_regression',
        'max_iter': 1000,
        'random_state': SEED,
        'class_weight': 'balanced'  # handles imbalance
    }
    mlflow.log_params(params)
    
    # train
    lr = LogisticRegression(
        max_iter=1000, 
        random_state=SEED, 
        class_weight='balanced'
    )
    lr.fit(X_train_processed, y_train)
    
    # predict
    y_pred = lr.predict(X_val_processed)
    y_pred_proba = lr.predict_proba(X_val_processed)[:, 1]
    
    # metrics
    metrics = {
        'auc_roc': round(roc_auc_score(y_val, y_pred_proba), 4),
        'f1': round(f1_score(y_val, y_pred), 4),
        'precision': round(precision_score(y_val, y_pred), 4),
        'recall': round(recall_score(y_val, y_pred), 4)
    }
    mlflow.log_metrics(metrics)
    
    # log model
    signature = infer_signature(X_train_processed, y_pred)
    mlflow.sklearn.log_model(lr, "model", signature=signature)
    
    run_id_lr = mlflow.active_run().info.run_id

print("=== LOGISTIC REGRESSION RESULTS ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")
print(f"\nRun ID: {run_id_lr}")

2026/05/22 09:08:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/22 09:08:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


=== LOGISTIC REGRESSION RESULTS ===
  auc_roc: 0.7839
  f1: 0.18
  precision: 0.1056
  recall: 0.6078

Run ID: 6facc2008eb941a19fac115302c846f2


In [7]:
# Cell 7 — MLflow Experiment 2: XGBoost Baseline

# scale_pos_weight handles class imbalance
# ratio of non-fraud to fraud
scale_pos_weight = int((y_train == 0).sum() / (y_train == 1).sum())
print(f"scale_pos_weight: {scale_pos_weight}")

with mlflow.start_run(run_name="xgboost-baseline"):
    
    # log feature metadata
    mlflow.log_params(feature_metadata)
    
    # model params
    params = {
        'model_type': 'xgboost',
        'n_estimators': 300,
        'max_depth': 6,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'scale_pos_weight': scale_pos_weight,
        'random_state': SEED,
        'eval_metric': 'auc',
        'early_stopping_rounds': 20
    }
    mlflow.log_params(params)
    
    # train
    xgb_model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=SEED,
        eval_metric='auc',
        early_stopping_rounds=20,
        verbosity=0
    )
    
    xgb_model.fit(
        X_train_processed, y_train,
        eval_set=[(X_val_processed, y_val)],
        verbose=50
    )
    
    # predict
    y_pred = xgb_model.predict(X_val_processed)
    y_pred_proba = xgb_model.predict_proba(X_val_processed)[:, 1]
    
    # metrics
    metrics = {
        'auc_roc': round(roc_auc_score(y_val, y_pred_proba), 4),
        'f1': round(f1_score(y_val, y_pred), 4),
        'precision': round(precision_score(y_val, y_pred), 4),
        'recall': round(recall_score(y_val, y_pred), 4)
    }
    mlflow.log_metrics(metrics)
    
    # log feature importance
    importance_df = pd.DataFrame({
        'feature': num_cols + cat_cols,
        'importance': xgb_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    importance_df.to_csv('feature_importance.csv', index=False)
    mlflow.log_artifact('feature_importance.csv')
    
    # log model
    signature = infer_signature(X_train_processed, y_pred)
    mlflow.xgboost.log_model(xgb_model, "model", signature=signature)
    
    run_id_xgb = mlflow.active_run().info.run_id

print("=== XGBOOST BASELINE RESULTS ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")
print(f"\nTop 10 important features:")
print(importance_df.head(10))
print(f"\nRun ID: {run_id_xgb}")

scale_pos_weight: 27
[0]	validation_0-auc:0.85014
[50]	validation_0-auc:0.89063
[100]	validation_0-auc:0.90715
[150]	validation_0-auc:0.91844
[200]	validation_0-auc:0.92551
[250]	validation_0-auc:0.93022
[299]	validation_0-auc:0.93405


2026/05/22 09:09:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


=== XGBOOST BASELINE RESULTS ===
  auc_roc: 0.9341
  f1: 0.3694
  precision: 0.239
  recall: 0.8137

Top 10 important features:
       feature  importance
97         V70    0.078456
14          C8    0.071397
118        V91    0.071166
117        V90    0.041835
194       V308    0.037726
203       V317    0.029184
20         C14    0.025415
10          C4    0.023312
169       V283    0.018665
208  ProductCD    0.015466

Run ID: fd73ad80c6a1408d8c75a4f2bce41b01


In [8]:
# Cell 8 — MLflow Experiment 3: XGBoost Tuned

with mlflow.start_run(run_name="xgboost-tuned"):
    
    mlflow.log_params(feature_metadata)
    
    params = {
        'model_type': 'xgboost_tuned',
        'n_estimators': 500,
        'max_depth': 8,
        'learning_rate': 0.02,
        'subsample': 0.8,
        'colsample_bytree': 0.6,
        'min_child_weight': 5,
        'gamma': 1,
        'reg_alpha': 0.1,
        'reg_lambda': 1.5,
        'scale_pos_weight': scale_pos_weight,
        'random_state': SEED,
        'eval_metric': 'auc',
        'early_stopping_rounds': 30
    }
    mlflow.log_params(params)
    
    xgb_tuned = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=8,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.6,
        min_child_weight=5,
        gamma=1,
        reg_alpha=0.1,
        reg_lambda=1.5,
        scale_pos_weight=scale_pos_weight,
        random_state=SEED,
        eval_metric='auc',
        early_stopping_rounds=30,
        verbosity=0
    )
    
    xgb_tuned.fit(
        X_train_processed, y_train,
        eval_set=[(X_val_processed, y_val)],
        verbose=50
    )
    
    y_pred = xgb_tuned.predict(X_val_processed)
    y_pred_proba = xgb_tuned.predict_proba(X_val_processed)[:, 1]
    
    metrics_tuned = {
        'auc_roc': round(roc_auc_score(y_val, y_pred_proba), 4),
        'f1': round(f1_score(y_val, y_pred), 4),
        'precision': round(precision_score(y_val, y_pred), 4),
        'recall': round(recall_score(y_val, y_pred), 4)
    }
    mlflow.log_metrics(metrics_tuned)
    
    importance_df_tuned = pd.DataFrame({
        'feature': num_cols + cat_cols,
        'importance': xgb_tuned.feature_importances_
    }).sort_values('importance', ascending=False)
    
    importance_df_tuned.to_csv('feature_importance_tuned.csv', index=False)
    mlflow.log_artifact('feature_importance_tuned.csv')
    
    signature = infer_signature(X_train_processed, y_pred)
    mlflow.xgboost.log_model(xgb_tuned, "model", signature=signature)
    
    run_id_xgb_tuned = mlflow.active_run().info.run_id

print("=== XGBOOST TUNED RESULTS ===")
for k, v in metrics_tuned.items():
    print(f"  {k}: {v}")
print(f"\nTop 10 important features:")
print(importance_df_tuned.head(10))
print(f"\nRun ID: {run_id_xgb_tuned}")

[0]	validation_0-auc:0.85566
[50]	validation_0-auc:0.89938
[100]	validation_0-auc:0.90656
[150]	validation_0-auc:0.91637
[200]	validation_0-auc:0.92328
[250]	validation_0-auc:0.92849
[300]	validation_0-auc:0.93322
[350]	validation_0-auc:0.93714
[400]	validation_0-auc:0.94020
[450]	validation_0-auc:0.94263
[499]	validation_0-auc:0.94483


2026/05/22 09:12:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


=== XGBOOST TUNED RESULTS ===
  auc_roc: 0.9448
  f1: 0.4186
  precision: 0.2812
  recall: 0.8185

Top 10 important features:
    feature  importance
97      V70    0.094874
14       C8    0.063007
118     V91    0.050166
117     V90    0.042919
194    V308    0.029185
96      V69    0.026232
10       C4    0.025078
20      C14    0.022145
203    V317    0.019607
180    V294    0.016759

Run ID: ab88b3fc0be7418ab77267d739eb4c48


In [9]:
# Cell 9 — Compare all MLflow runs, pick best model

import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name("fraud-detection")

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.auc_roc DESC"]
)

print("=== ALL EXPERIMENT RUNS ===")
print(f"{'Run Name':<30} {'AUC':>6} {'F1':>6} {'Recall':>8} {'Precision':>10}")
print("-" * 65)
for run in runs:
    name = run.data.tags.get('mlflow.runName', 'unknown')
    auc = run.data.metrics.get('auc_roc', 0)
    f1 = run.data.metrics.get('f1', 0)
    recall = run.data.metrics.get('recall', 0)
    precision = run.data.metrics.get('precision', 0)
    print(f"{name:<30} {auc:>6.4f} {f1:>6.4f} {recall:>8.4f} {precision:>10.4f}")

# best run
best_run = runs[0]
best_run_id = best_run.info.run_id
best_run_name = best_run.data.tags.get('mlflow.runName')

print(f"\n=== BEST MODEL ===")
print(f"Run: {best_run_name}")
print(f"Run ID: {best_run_id}")
print(f"AUC: {best_run.data.metrics['auc_roc']}")

=== ALL EXPERIMENT RUNS ===
Run Name                          AUC     F1   Recall  Precision
-----------------------------------------------------------------
xgboost-tuned                  0.9448 0.4186   0.8185     0.2812
xgboost-baseline               0.9341 0.3694   0.8137     0.2390
logistic-regression-baseline   0.7839 0.1800   0.6078     0.1056
test-run                       0.0000 0.0000   0.0000     0.0000

=== BEST MODEL ===
Run: xgboost-tuned
Run ID: ab88b3fc0be7418ab77267d739eb4c48
AUC: 0.9448


In [10]:
# Cell 10 — SHAP Explainability on Best Model

import shap

# use 1000 samples for speed
X_shap = X_val_processed[:1000]

explainer = shap.TreeExplainer(xgb_tuned)
shap_values = explainer.shap_values(X_shap)

# get feature names in correct order
feature_names = num_cols + cat_cols

# top 15 features by mean absolute SHAP value
shap_importance = pd.DataFrame({
    'feature': feature_names,
    'shap_importance': np.abs(shap_values).mean(axis=0)
}).sort_values('shap_importance', ascending=False)

print("=== TOP 15 FEATURES BY SHAP ===")
print(shap_importance.head(15).to_string(index=False))

# save shap importance for MLflow
shap_importance.to_csv('shap_importance.csv', index=False)

# log shap to best run
with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact('shap_importance.csv')

print("\nSHAP importance logged to MLflow best run")

=== TOP 15 FEATURES BY SHAP ===
       feature  shap_importance
TransactionAmt         0.210157
           C13         0.206139
            C1         0.199498
           C14         0.197497
         card6         0.178224
            C8         0.137809
           V70         0.134878
            M4         0.120683
         card1         0.118498
           C11         0.111179
            C2         0.110621
         card2         0.107349
            D4         0.106430
 P_emaildomain         0.105077
            D1         0.102347

SHAP importance logged to MLflow best run


In [12]:
# Cell 11 — Save model + preprocessor + reference data

import pickle
import json

# save preprocessor
with open('preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

# save model
with open('model.pkl', 'wb') as f:
    pickle.dump(xgb_tuned, f)

# save reference data (sample of training data for drift detection)
reference_data = X_train.sample(5000, random_state=SEED).copy()
reference_data['isFraud'] = y_train[reference_data.index]
reference_data.to_csv('reference_data.csv', index=False)

# save feature metadata
feature_metadata['best_run_id'] = best_run_id
feature_metadata['best_model'] = 'xgboost-tuned'
feature_metadata['auc_roc'] = 0.9448
feature_metadata['num_cols'] = num_cols
feature_metadata['cat_cols'] = cat_cols

with open('feature_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)



print("=== SAVED FILES ===")
import os
for f in ['model.pkl', 'preprocessor.pkl', 'reference_data.csv', 'feature_metadata.json']:
    size = os.path.getsize(f) / 1024**2
    print(f"  {f}: {size:.2f} MB")

print("\nAll artifacts logged to MLflow")
print("\nDownload these 4 files from Kaggle output panel:")
print("  1. model.pkl")
print("  2. preprocessor.pkl")
print("  3. reference_data.csv")
print("  4. feature_metadata.json")

=== SAVED FILES ===
  model.pkl: 4.81 MB
  preprocessor.pkl: 0.01 MB
  reference_data.csv: 4.05 MB
  feature_metadata.json: 0.01 MB

All artifacts logged to MLflow

Download these 4 files from Kaggle output panel:
  1. model.pkl
  2. preprocessor.pkl
  3. reference_data.csv
  4. feature_metadata.json
